# Part 1: Training an implicit transfer operator (ITO) on the Wolfe–Quapp potential

A molecular dynamics (MD) simulation takes very small time steps, so rare events such as conformational changes need
enormous numbers of steps. An **implicit transfer operator (ITO)** learns the *transition density*

$$p(x_{t+\tau} \mid x_t, \tau)$$

for large lag times $\tau$, using a generative model that we can sample from. One sample is one big time step,
and a single model covers many lag times $\tau$.

In this notebook you will

1. simulate **reference trajectories** of a 2D toy system (overdamped Langevin dynamics on the Wolfe–Quapp potential),
2. **train an ITO** with conditional flow matching on pairs $(x_t, x_{t+\tau})$,
3. **evaluate** the model the way you would evaluate any MD surrogate:
   free energies, stationarity, implied timescales and the Chapman–Kolmogorov test.

Run the cells from top to bottom (`Shift + Enter`). Running all cells takes about 5 minutes on a laptop.
Cells marked **Question** are for thinking and discussing with your neighbour. **Exercises** suggest changes to try once you have
run through the notebook once.

*Reference:* Schreiner, Winther & Olsson, *Implicit Transfer Operator Learning: Multiple Time-Resolution Surrogates for
Molecular Dynamics*, NeurIPS 2023 ([arXiv:2305.18046](https://arxiv.org/abs/2305.18046)).

In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import deeptime as dt

from minito.systems import WolfeQuapp, simulate
from minito.ito import LaggedPairs, VectorField, sample, rollout, save_model, load_model
from minito import analysis as an

torch.manual_seed(0)
print("torch", torch.__version__, "| threads:", torch.get_num_threads())

## 1. The system

The Wolfe–Quapp potential is

$$E(x, y) = x^4 + y^4 - 2x^2 - 4y^2 + xy + 0.3x + 0.1y .$$

A particle moves according to overdamped Langevin dynamics at temperature $k_BT = 1$:

$$x_{n+1} = x_n - \nabla E(x_n)\,\Delta t + \sqrt{2 k_BT \Delta t}\;\xi_n, \qquad \xi_n \sim \mathcal{N}(0, I).$$

The equilibrium (Boltzmann) distribution is $\pi(x) \propto e^{-E(x)/k_BT}$. Because the system is 2D, we can compute the
**exact** free energy along $x$ or $y$, $F(y) = -k_BT \ln \int e^{-E(x,y)/k_BT}\,dx$, and use it as ground truth later.

In [ ]:
system = WolfeQuapp()
kT = 1.0

X, Y, E = system.grid()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cs = axes[0].contourf(X, Y, E, levels=np.arange(-7, 3, 0.5), cmap="viridis")
plt.colorbar(cs, ax=axes[0], label="E")
for (mx, my), name in zip(system.minima, an.BASIN_NAMES):
    axes[0].plot(mx, my, "w*", ms=12)
    axes[0].annotate(name, (mx, my), textcoords="offset points", xytext=(-25, 10), color="w")
axes[0].set(xlabel="x", ylabel="y", title="Wolfe-Quapp potential", aspect="equal")
for ax, axis, name in [(axes[1], 0, "x"), (axes[2], 1, "y")]:
    q, F = system.free_energy_1d(kT, axis=axis)
    ax.plot(q, F, "k")
    ax.set(xlabel=name, ylabel=f"F({name}) / kT", title=f"exact free energy along {name}", ylim=(0, 10))
plt.tight_layout()

**Question.** The two saddle points between the top and bottom halves are at $(-1.02, -0.12)$ with $E = -1.25$ and
$(0.94, 0.13)$ with $E = -0.64$. The minima have $E = -6.76$ (top-left), $-6.37$ (bottom-right) and $-4.14$ (bottom-left).
Which transitions do you expect to be slow at $k_BT = 1$? Roughly how much slower than the others?

## 2. Reference simulations

We run 100 independent walkers ("trajectories") in parallel:

* a **training set**: 100 trajectories of 500 time units, used to train the ITO,
* an independent **reference set**: 100 trajectories of 1000 time units, used only for evaluation.

We integrate with $\Delta t = 0.002$ and save every 50 steps, so consecutive **frames are 0.1 time units apart**.
All lag times below are given in frames.

In [ ]:
DT, STRIDE = 2e-3, 50
FRAME = DT * STRIDE  # time between saved frames (0.1 time units)

rng = np.random.default_rng(0)
t0 = time.time()
train_trajs = simulate(system, system.minima[rng.integers(0, 3, 100)], kT=kT, dt=DT, n_steps=250_000, stride=STRIDE, seed=1)
ref_trajs = simulate(system, system.minima[rng.integers(0, 3, 100)], kT=kT, dt=DT, n_steps=500_000, stride=STRIDE, seed=2)
print(f"simulation took {time.time() - t0:.0f} s")
print("training set :", train_trajs.shape, "(trajectories, frames, dim)")
print("reference set:", ref_trajs.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw=dict(width_ratios=[2.5, 1]))
t = np.arange(ref_trajs.shape[1]) * FRAME
for i in range(3):
    axes[0].plot(t, ref_trajs[i, :, 1], lw=0.5)
axes[0].set(xlabel="time", ylabel="y", title="y(t) for 3 reference trajectories")
an.plot_potential(axes[1], system)
axes[1].plot(*ref_trajs[0, :2000].T, ".", ms=1, alpha=0.5)
axes[1].set_title("trajectory 0, first 200 time units")
plt.tight_layout()

n_switch = np.abs(np.diff(an.basin(ref_trajs) == 0, axis=1)).sum(axis=1)
print(f"top <-> bottom transitions per trajectory (1000 time units): mean {n_switch.mean():.1f}")

## 3. Training data: lagged pairs

The ITO is trained on pairs $(x_t, x_{t+\tau})$ cut from the training trajectories. Each pair comes with its lag $\tau$, which is
drawn log-uniformly between 1 and `MAX_LAG` frames, so short and long lags are equally represented. All lags are learned by **one**
network that takes $\tau$ as an input.

Below we look directly at the data: we take all reference frames that are close to the bottom-left minimum and plot where these
trajectories are $\tau$ frames later. This is the **transition density** the ITO has to learn.

In [ ]:
MAX_LAG = 500  # frames = 50 time units
pairs = LaggedPairs(train_trajs, max_lag=MAX_LAG)
x_t, x_tau, lag = pairs.sample(5)
print("x_t   =", x_t.numpy().round(2).tolist())
print("x_tau =", x_tau.numpy().round(2).tolist())
print("lag   =", lag.tolist())

In [ ]:
def md_transitions(trajs, x0, lag, radius=0.15):
    # Frames within `radius` of x0, and where the same trajectories are `lag` frames later.
    start = trajs[:, :-lag]
    near = np.linalg.norm(start - x0, axis=-1) < radius
    return trajs[:, lag:][near]


x0 = system.minima[2]  # the bottom-left minimum
LAGS_TO_SHOW = [1, 10, 100, 500]
fig, axes = plt.subplots(1, len(LAGS_TO_SHOW), figsize=(16, 4))
for ax, lag in zip(axes, LAGS_TO_SHOW):
    an.plot_potential(ax, system)
    end = md_transitions(ref_trajs, x0, lag)
    ax.plot(*end[:3000].T, ".", ms=2, alpha=0.3)
    ax.plot(*x0, "r*", ms=12)
    ax.set_title(f"MD: lag {lag} frames = {lag * FRAME:g} time units")
plt.tight_layout()

**Question.** How does the transition density change with the lag? What does it look like for a lag much longer than the slowest
relaxation in the system?

## 4. The model: conditional flow matching

We need a generative model for $p(x_{t+\tau} \mid x_t, \tau)$. We use **flow matching**:

* draw noise $z_0 \sim \mathcal{N}(0, I)$, and let $x_1 = x_{t+\tau}$ be the data point,
* the straight path between them is $z_s = s\,x_1 + (1 - s)\,z_0$, for $s \in [0, 1]$, with velocity $x_1 - z_0$,
* a neural network $v_\theta(z_s, s \mid x_t, \tau)$ is trained to predict that velocity:

$$\mathcal{L}(\theta) = \mathbb{E}_{(x_t, x_{t+\tau}, \tau),\, z_0,\, s}\;\big\| v_\theta(z_s, s \mid x_t, \tau) - (x_{t+\tau} - z_0) \big\|^2 .$$

To **sample** $x_{t+\tau}$ we draw new noise and integrate the ODE $dz/ds = v_\theta(z, s \mid x_t, \tau)$ from $s = 0$ to $1$.
Conditioning on $x_t$ and $\tau$ makes this a model of the transition density, which is what makes it an ITO.

The network is a small MLP (see `minito/ito.py`). Its inputs are the noisy point $z_s$, the current state $x_t$, and Fourier
features of the flow time $s$ and of $\log \tau$. The loss is short enough to write out here:

In [ ]:
model = VectorField(dim=2, hidden=256, n_layers=4, max_lag=MAX_LAG)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")


def flow_matching_loss(model, x_t, x_tau, lag):
    z0 = torch.randn_like(x_tau)                         # noise
    s = torch.rand(x_tau.shape[0])                       # flow time in [0, 1]
    z_s = s[:, None] * x_tau + (1 - s[:, None]) * z0     # point on the straight path noise -> data
    target = x_tau - z0                                  # velocity along that path
    return ((model(z_s, s, x_t, lag) - target) ** 2).mean()

## 5. Training

With the default settings, training takes about 1–3 minutes on a laptop CPU. If your laptop is very slow, set `USE_PRETRAINED = True`
to load a model that was trained the same way, for longer (30,000 iterations).

In [ ]:
N_ITER = 10_000
BATCH_SIZE = 1024
LR = 1e-3
USE_PRETRAINED = False

if USE_PRETRAINED:
    model, _ = load_model("../data/wolfe_quapp/ito_pretrained.pt")
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_ITER)
    losses = []
    t0 = time.time()
    model.train()
    for it in range(N_ITER):
        x_t, x_tau, lag = pairs.sample(BATCH_SIZE)
        loss = flow_matching_loss(model, x_t, x_tau, lag)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
        if (it + 1) % 1000 == 0:
            print(f"iteration {it + 1:6d}   loss {np.mean(losses[-1000:]):.4f}   ({time.time() - t0:.0f} s)")
    model.eval()
    save_model(model, "my_ito.pt")

    plt.figure(figsize=(6, 3))
    plt.plot(np.convolve(losses, np.ones(100) / 100, mode="valid"))
    plt.xlabel("iteration")
    plt.ylabel("loss (running mean)")

**Question.** The loss levels off well above zero. Why can flow matching never reach zero loss, even with a perfect model?
What does that mean for using the loss value to judge model quality?

## 6. Sampling from the ITO

We now sample $x_{t+\tau} \sim p_\theta(\cdot \mid x_t = x_0, \tau)$ from the bottom-left minimum and compare with the MD transitions
from Section 3. `ODE_STEPS` is the number of Euler steps used to integrate the flow ODE. Each ITO sample costs `ODE_STEPS` network
evaluations, *regardless of the lag*.

In [ ]:
ODE_STEPS = 50

fig, axes = plt.subplots(2, len(LAGS_TO_SHOW), figsize=(16, 8))
for j, lag in enumerate(LAGS_TO_SHOW):
    ito_end = sample(model, np.tile(x0, (3000, 1)), lag, ODE_STEPS).numpy()
    md_end = md_transitions(ref_trajs, x0, lag)[:3000]
    for i, (pts, name) in enumerate([(md_end, "MD"), (ito_end, "ITO")]):
        an.plot_potential(axes[i, j], system)
        axes[i, j].plot(*pts.T, ".", ms=2, alpha=0.3, color=f"C{i}")
        axes[i, j].plot(*x0, "r*", ms=12)
        axes[i, j].set_title(f"{name}: lag {lag} frames")
plt.tight_layout()

Because one model covers all lags, we can **roll out** trajectories by repeatedly sampling at a fixed lag. Below, one ITO
trajectory with lag 10 frames (1 time unit) is shown next to an MD trajectory.

In [ ]:
ito_traj = rollout(model, x0[None], lag=10, n_steps=1000, ode_steps=ODE_STEPS)[0]
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(np.arange(ref_trajs.shape[1]) * FRAME, ref_trajs[0, :, 1], lw=0.5, label="MD (frame every 0.1)")
ax.plot(np.arange(len(ito_traj)) * 10 * FRAME, ito_traj[:, 1], lw=0.8, label="ITO (step = 1 time unit)")
ax.set(xlabel="time", ylabel="y", xlim=(0, 1000))
ax.legend(loc="upper right");

## 7. Evaluation I: free energies and stationarity

**Free energies.** We roll out 1000 ITO trajectories from the bottom-left minimum with a large lag (100 frames = 10 time units),
discard the first 10 steps as equilibration, and compare the sampled free energies with the MD reference and with the exact result.

In [ ]:
t0 = time.time()
ito_long = rollout(model, np.tile(x0, (1000, 1)), lag=100, n_steps=60, ode_steps=ODE_STEPS)
print(f"rollout took {time.time() - t0:.0f} s")
ito_eq = ito_long[:, 10:].reshape(-1, 2)   # drop equilibration
md_eq = ref_trajs[:, 100:].reshape(-1, 2)   # drop the first 10 time units

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
bins = np.linspace(-2.2, 2.2, 45)
for k, (pts, name) in enumerate([(md_eq, "MD"), (ito_eq, "ITO")]):
    xe, ye, F = an.free_energy_2d(pts, kT, bins)
    cs = axes[k].pcolormesh(xe, ye, F, vmax=8, cmap="viridis")
    axes[k].set(title=f"{name}: F(x, y) / kT", xlabel="x", ylabel="y", aspect="equal")
for ax, axis, name in [(axes[2], 0, "x"), (axes[3], 1, "y")]:
    ax.plot(*system.free_energy_1d(kT, axis=axis), "k", label="exact")
    ax.plot(*an.free_energy_1d(md_eq[:, axis], kT, bins), "o", ms=4, label="MD")
    ax.plot(*an.free_energy_1d(ito_eq[:, axis], kT, bins), "s", ms=4, mfc="none", label="ITO")
    ax.set(xlabel=name, ylabel=f"F({name}) / kT", ylim=(0, 8))
    ax.legend()
plt.tight_layout()

print("basin populations   ", "   ".join(f"{n:>12s}" for n in an.BASIN_NAMES))
print("MD                  ", "   ".join(f"{p:12.3f}" for p in an.basin_populations(md_eq)))
print("ITO                 ", "   ".join(f"{p:12.3f}" for p in an.basin_populations(ito_eq)))

**Stationarity.** If $x_t$ is drawn from the equilibrium distribution $\pi$, then $x_{t+\tau}$ must also follow $\pi$, for **every**
lag. We take 20,000 equilibrium samples from the reference MD, apply **one** ITO step, and check that the basin populations do not change.

We also look at the **relaxation** towards equilibrium: all walkers start in the bottom-left minimum, and we follow the basin populations
over time for MD (fresh simulation, 1000 walkers) and for the ITO (lag 10 frames).

In [ ]:
x_eq = md_eq[np.random.default_rng(1).choice(len(md_eq), 20_000, replace=False)]
print("lag (frames)    ", "   ".join(f"{n:>12s}" for n in an.BASIN_NAMES))
print("before          ", "   ".join(f"{p:12.3f}" for p in an.basin_populations(x_eq)))
for lag in [1, 10, 100, 500]:
    x_next = sample(model, x_eq, lag, ODE_STEPS).numpy()
    print(f"after {lag:4d}      ", "   ".join(f"{p:12.3f}" for p in an.basin_populations(x_next)))

In [ ]:
n_steps = 60  # 60 time units
md_relax = simulate(system, np.tile(x0, (1000, 1)), kT=kT, dt=DT, n_steps=int(n_steps / DT), stride=STRIDE, seed=3)
ito_relax = rollout(model, np.tile(x0, (1000, 1)), lag=10, n_steps=n_steps, ode_steps=ODE_STEPS)

fig, ax = plt.subplots(figsize=(7, 4))
t_md = np.arange(1, md_relax.shape[1] + 1) * FRAME
t_ito = np.arange(ito_relax.shape[1]) * 10 * FRAME
for b in range(3):
    ax.plot(t_md, (an.basin(md_relax) == b).mean(0), color=f"C{b}", label=f"MD {an.BASIN_NAMES[b]}")
    ax.plot(t_ito, (an.basin(ito_relax) == b).mean(0), "o", mfc="none", color=f"C{b}", label=f"ITO {an.BASIN_NAMES[b]}")
ax.set(xlabel="time", ylabel="population", title="relaxation from the bottom-left minimum")
ax.legend(ncol=2, fontsize=8);

**Question.** The relaxation curves show at least two time scales. Which processes do they correspond to?

## 8. Evaluation II: implied timescales

We discretise the plane into 40 states with k-means and build **Markov state models (MSMs)**. The transition matrix $T(\tau)$ at
lag $\tau$ has eigenvalues $\lambda_i(\tau)$, and the **implied timescales** are

$$t_i = -\frac{\tau}{\ln \lambda_i(\tau)} .$$

For Markovian dynamics, $t_i$ does not depend on $\tau$.

* **MD:** count transitions between states in the reference trajectories, as in a standard MSM analysis.
* **ITO:** take 20,000 equilibrium start points, propagate each with **one** ITO sample at lag $\tau$, and count transitions.
  This gives the transition matrix that the ITO implies at each lag, discretised in exactly the same way as the MD data.

In [ ]:
states = an.make_states(ref_trajs, n_states=40)
dtrajs = [states.transform(t) for t in ref_trajs]
MSM_LAGS = [1, 2, 5, 10, 20, 50, 100, 200]  # frames

t0 = time.time()
md_msms = [an.msm_from_trajectories(dtrajs, lag) for lag in MSM_LAGS]
ito_msms = [an.msm_from_ito(model, states, x_eq, lag, ODE_STEPS) for lag in MSM_LAGS]
print(f"took {time.time() - t0:.0f} s")

fig, ax = plt.subplots(figsize=(7, 5))
lag_t = np.array(MSM_LAGS) * FRAME
for i in range(3):
    ax.plot(lag_t, [m.timescales(3)[i] * FRAME for m in md_msms], "-", color=f"C{i}", label=f"MD $t_{i+1}$")
    ax.plot(lag_t, [m.timescales(3)[i] * FRAME for m in ito_msms], "o--", mfc="none", color=f"C{i}", label=f"ITO $t_{i+1}$")
ax.fill_between(lag_t, 1e-2, lag_t, color="0.85")
ax.set(xscale="log", yscale="log", xlabel="lag time", ylabel="implied timescale", ylim=(0.1, 200))
ax.legend(ncol=2);

**Question.** The MD timescales increase with the lag at short lags. Why? Do the ITO timescales show the same trend, and should they?
What is the slowest timescale, and how does it compare with your estimate from the barrier heights?

*(In the grey area the timescale is shorter than the lag time and cannot be resolved.)*

## 9. Evaluation III: Chapman–Kolmogorov test

Markovian dynamics satisfy $T(k\tau) = T(\tau)^k$: propagating $k$ times with lag $\tau$ must give the same result as propagating
once with lag $k\tau$. For the ITO this is not automatic. The model has learned a separate conditional distribution for every
$\tau$, and nothing in the training forces them to be consistent. The CK test checks this self-consistency.

We use the standard deeptime CK test with two metastable sets (from PCCA+). Panel $i \to j$ shows the probability of being in set $j$
after time $k\tau$ when starting in set $i$. Black lines are direct estimates at lag $k\tau$, and dashed lines are predictions
$T(\tau)^k$ from the shortest lag ($\tau$ = 10 frames).

In [ ]:
CK_LAGS = [10, 50, 100, 200, 300, 400, 500]  # frames (all multiples of the first)
md_ck_models = [an.msm_from_trajectories(dtrajs, lag) for lag in CK_LAGS]
ito_ck_models = [an.msm_from_ito(model, states, x_eq, lag, ODE_STEPS) for lag in CK_LAGS]

ck_md = md_ck_models[0].ck_test(md_ck_models, n_metastable_sets=2)
ck_ito = ito_ck_models[0].ck_test(ito_ck_models, n_metastable_sets=2)
from matplotlib.lines import Line2D

for ck, name in [(ck_md, "MD"), (ck_ito, "ITO")]:
    grid = dt.plots.plot_ck_test(ck, legend=False, xlabel="lag time (frames)")
    handles = [Line2D([], [], color="k"), Line2D([], [], color="C0", ls="--")]
    grid.figure.legend(handles, [r"estimate at lag $k\tau$", r"prediction $T(\tau)^k$"], title=f"CK test: {name}",
                       loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.08))

We can also check the Chapman–Kolmogorov property directly on samples, without any discretisation: from the bottom-left minimum,
compare **one** ITO step with lag $2\tau$ against **two** ITO steps with lag $\tau$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tau in zip(axes, [5, 25, 100]):
    start = np.tile(x0, (20_000, 1))
    one_step = sample(model, start, 2 * tau, ODE_STEPS).numpy()
    two_steps = sample(model, sample(model, start, tau, ODE_STEPS), tau, ODE_STEPS).numpy()
    md_ref = md_transitions(ref_trajs, x0, 2 * tau)
    bins = np.linspace(-2.2, 2.2, 60)
    ax.hist(md_ref[:, 1], bins, density=True, histtype="stepfilled", color="0.8", label="MD")
    ax.hist(one_step[:, 1], bins, density=True, histtype="step", lw=2, label=f"ITO: 1 step of {2 * tau}")
    ax.hist(two_steps[:, 1], bins, density=True, histtype="step", lw=2, ls="--", label=f"ITO: 2 steps of {tau}")
    ax.set(xlabel="y", title=f"y after {2 * tau} frames")
    ax.legend(fontsize=8)
plt.tight_layout()

## 10. Exercises

Pick a few and discuss what you find. Re-run Sections 4–9 after each change, and for each one ask
*which evaluation is the first to show a problem?*

1. **ODE steps.** Set `ODE_STEPS = 5`, `10` and `20` and re-run Sections 6–9. What happens to the free energies and the timescales? Why?
2. **Training budget.** Train for only 1000 or 3000 iterations. Does the loss curve tell you the model is worse? Which evaluation does?
3. **Lag range.** Train with `MAX_LAG = 10` (1 time unit). Can rollouts at lag 10 still reproduce the slow timescale and the free energy?
   What happens if you ask this model for a sample at lag 500?
4. **Short trajectories.** Train on short pieces of trajectory only (20 time units each, shorter than the slowest timescale).
   Can the ITO still learn the slow process?
   ```python
   MAX_LAG = 100
   pairs = LaggedPairs(train_trajs[:, :200], max_lag=MAX_LAG)
   ```
5. **Short, non-equilibrium data.** Train on 1000 trajectories of 10 time units that all start in the bottom-left minimum, so the
   data are far from equilibrium. Does the ITO still get the right equilibrium populations? Why could it, at least in principle?
   ```python
   MAX_LAG = 50
   local_trajs = simulate(system, np.tile(system.minima[2], (1000, 1)), kT=kT, dt=DT, n_steps=5_000, stride=STRIDE, seed=4)
   pairs = LaggedPairs(local_trajs, max_lag=MAX_LAG)
   ```
6. **Temperature.** Set `kT = 0.8` in Section 1, re-run everything and compare the slowest timescale with the one at $k_BT = 1$.
   Does the ratio agree with the Arrhenius estimate $e^{\Delta E\,(1/0.8 - 1)}$ for a barrier $\Delta E \approx 5$?
7. **Model size.** Use `hidden=32` or `n_layers=2`. Where does the smaller model fail first?

**Wrap-up questions**

* In this toy system one ITO step costs about as much as a few hundred MD steps. When does an ITO pay off?
* Which of the evaluations above would you still be able to run for a protein, where no long reference simulation exists? (→ Part 2)